# ECE 5831 – Assignment 09 (Part 2): Text Processing (`TextProcess` Class)
**Name:** Michel Ludovic Temgoua  
**Course:** ECE 5831 – Pattern Recognition and Neural Networks  

## 1. Overview & Tokenization Design
Neural networks operate on numerical tensors rather than raw strings. In this notebook, we implement the **`TextProcess`** class to perform end-to-end text vectorization and reconstruction:
- **Standardization & Tokenization:** Lowercases input text, separates punctuation, and splits text into discrete word tokens.
- **Vocabulary Indexing (`word_to_idx` & `idx_to_word`):** Maps each unique vocabulary token to an integer ID, reserving index `0` for `<PAD>` (sequence padding) and index `1` for `<UNK>` (out-of-vocabulary words).
- **Encoding (`encode`):** Converts raw sentences into fixed-length or variable-length integer sequences.
- **Decoding (`decode`):** Reconstructs integer token sequences back into human-readable text strings.

In [1]:
import re
from collections import Counter
import numpy as np


class TextProcess:
    """Handles vocabulary building, numerical text encoding, and text decoding."""

    def __init__(self, max_vocab_size=None, lowercase=True):
        self.max_vocab_size = max_vocab_size
        self.lowercase = lowercase
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"
        self.word_to_idx = {self.pad_token: 0, self.unk_token: 1}
        self.idx_to_word = {0: self.pad_token, 1: self.unk_token}

    def standardize_and_tokenize(self, text):
        """Standardizes text and splits words while preserving punctuation as tokens."""
        if self.lowercase:
            text = text.lower()
        # Separate punctuation marks so words like 'learning.' tokenize cleanly
        text = re.sub(r"([.,!?;:()\"'])", r" \1 ", text)
        tokens = text.strip().split()
        return tokens

    def fit(self, corpus):
        """Builds bidirectional word <-> index mappings from a list of text documents."""
        counter = Counter()
        for doc in corpus:
            counter.update(self.standardize_and_tokenize(doc))

        most_common = counter.most_common(
            None if self.max_vocab_size is None else max(0, self.max_vocab_size - 2)
        )

        self.word_to_idx = {self.pad_token: 0, self.unk_token: 1}
        for idx, (word, _) in enumerate(most_common, start=2):
            self.word_to_idx[word] = idx

        self.idx_to_word = {idx: word for word, idx in self.word_to_idx.items()}
        return self

    def encode(self, text, max_length=None):
        """Converts a raw text string into a list of numerical token IDs."""
        tokens = self.standardize_and_tokenize(text)
        encoded = [self.word_to_idx.get(tok, self.word_to_idx[self.unk_token]) for tok in tokens]

        if max_length is not None:
            if len(encoded) < max_length:
                encoded = encoded + [self.word_to_idx[self.pad_token]] * (max_length - len(encoded))
            else:
                encoded = encoded[:max_length]
        return encoded

    def encode_batch(self, texts, max_length=None):
        """Encodes a list of strings into a 2D NumPy integer array."""
        encoded_list = [self.encode(t) for t in texts]
        if max_length is None:
            max_length = max(len(seq) for seq in encoded_list) if encoded_list else 0
        padded = [
            seq[:max_length] + [self.word_to_idx[self.pad_token]] * max(0, max_length - len(seq))
            for seq in encoded_list
        ]
        return np.array(padded, dtype=np.int32)

    def decode(self, token_ids, skip_pad=True):
        """Decodes a sequence of numerical token IDs back into readable text."""
        words = []
        for idx in token_ids:
            idx = int(idx)
            if skip_pad and idx == self.word_to_idx[self.pad_token]:
                continue
            words.append(self.idx_to_word.get(idx, self.unk_token))

        # Rejoin tokens and clean spacing before punctuation
        decoded = " ".join(words)
        decoded = re.sub(r"\s+([.,!?;:)])", r"\1", decoded)
        decoded = re.sub(r"\(\s+", "(", decoded)
        return decoded

## 2. Round-Trip Verification (`Text -> Numerical Representation -> Readable Text`)
We fit `TextProcess` on a sample corpus, inspect the learned vocabulary dictionary, and verify that:
1. Known sentences encode into integer vectors and decode cleanly back into readable text.
2. Unseen words gracefully map to `<UNK>` (`1`) without raising key errors.
3. Batch encoding produces a padded 2D NumPy array ready for an Embedding layer, and `decode` strips `<PAD>` (`0`) tokens when reconstructing text.

In [2]:
# Sample corpus for building vocabulary
corpus = [
    "Deep learning models learn hierarchical representations from data.",
    "Transfer learning improves image classification on limited datasets!",
    "Text processing converts readable sentences into numerical sequences.",
    "Neural networks use embedding layers to represent discrete word tokens.",
]

# 1. Initialize and fit TextProcess
processor = TextProcess()
processor.fit(corpus)

print(f"Vocabulary Size (including <PAD> and <UNK>): {len(processor.word_to_idx)}")
print("Sample Vocabulary Mapping:", dict(list(processor.word_to_idx.items())[:12]))
print("-" * 70)

# 2. Single-sentence encode & decode round-trip
sample_sentence = "Transfer learning improves image classification on limited datasets!"
encoded_ids = processor.encode(sample_sentence)
decoded_text = processor.decode(encoded_ids)

print("Original Text :", sample_sentence)
print("Encoded IDs   :", encoded_ids)
print("Decoded Text  :", decoded_text)
print("-" * 70)

# 3. Out-of-vocabulary (<UNK>) and fixed-length padding (<PAD>) test
unseen_sentence = "Transformers and recurrent networks process text sequences."
encoded_padded = processor.encode(unseen_sentence, max_length=12)
decoded_unseen = processor.decode(encoded_padded, skip_pad=True)

print("Unseen Text          :", unseen_sentence)
print("Encoded (max_len=12) :", encoded_padded)
print("Decoded (skip_pad)   :", decoded_unseen)
print("-" * 70)

# 4. Batch encoding into a 2D numerical tensor (ready for Keras Embedding layer)
batch_texts = [
    "Deep learning models learn from data.",
    "Text processing converts sentences into numerical sequences.",
]
batch_matrix = processor.encode_batch(batch_texts)
print("Batch Encoded Tensor Shape:", batch_matrix.shape)
print(batch_matrix)
for i, row in enumerate(batch_matrix):
    print(f"Decoded Row {i}: {processor.decode(row)}")

Vocabulary Size (including <PAD> and <UNK>): 37
Sample Vocabulary Mapping: {'<PAD>': 0, '<UNK>': 1, '.': 2, 'learning': 3, 'deep': 4, 'models': 5, 'learn': 6, 'hierarchical': 7, 'representations': 8, 'from': 9, 'data': 10, 'transfer': 11}
----------------------------------------------------------------------
Original Text : Transfer learning improves image classification on limited datasets!
Encoded IDs   : [11, 3, 12, 13, 14, 15, 16, 17, 18]
Decoded Text  : transfer learning improves image classification on limited datasets!
----------------------------------------------------------------------
Unseen Text          : Transformers and recurrent networks process text sequences.
Encoded (max_len=12) : [1, 1, 1, 28, 1, 19, 26, 2, 0, 0, 0, 0]
Decoded (skip_pad)   : <UNK> <UNK> <UNK> networks <UNK> text sequences.
----------------------------------------------------------------------
Batch Encoded Tensor Shape: (2, 8)
[[ 4  3  5  6  9 10  2  0]
 [19 20 21 23 24 25 26  2]]
Decoded Row 0: dee